# 73 — Run Blind-A with config 180 (wRRF + fine-tuned BGE-M3, Submission 1) → submission zip

Produces `prediction.json` for CodaBench by running the **Submission 1** pipeline
over the 80 Blind-A queries:

```
wRRF(BM25 + dense_lyrics + fine-tuned BGE-M3) → ProRank → v5-kto Qwen-3B responder
```

**Config 180** points at `OrRim123/recsys2026-bge-m3-music-v1-merged` (the model
trained in nb 70). CMQR is OFF for a clean read on the fine-tune's standalone
contribution; re-enable in Submission 2/3 after Stage A signal is established.

Per spec §9 (`docs/superpowers/specs/2026-05-18-ndcg-stretch-design.md`),
Submission 1 gate is **composite ≥ 0.21** (no regression vs the v5-kto baseline).
Expected wallclock: ~35–65 min on Blackwell.

> SID is dropped (per memory `project_sid_closed_2026_05_18`). This config uses
> the post-SID three-stream wRRF (BM25 + dense_lyrics + BGE-M3-FT).

In [ ]:
# 1) Setup (same pattern as notebook 62).
import os
from google.colab import userdata, drive
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
drive.mount('/content/drive', force_remount=False)

BRANCH = 'fresh-model'
!rm -rf /content/recsys2026
!git clone -b {BRANCH} https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026
%cd /content/recsys2026

DRIVE_BASE = '/content/drive/MyDrive'
LOCAL_BASE = '/content/recsys2026/experiments/cache'
os.makedirs(LOCAL_BASE, exist_ok=True)
for name, drive_subdir in [
    ('sid', 'recsys2026_sid_cache'),
    ('dense', 'recsys2026_dense_cache'),
    # Stage A fine-tuned BGE-M3 catalog pickle: written by nb 70 cell 6
    # to MyDrive/recsys2026_retrieval_v2_cache/dense_local/. Without this
    # symlink, DENSE_LOCAL FileNotFoundError on inference.
    ('dense_local', 'recsys2026_retrieval_v2_cache/dense_local'),
]:
    src = f'{DRIVE_BASE}/{drive_subdir}'
    dst = f'{LOCAL_BASE}/{name}'
    os.makedirs(src, exist_ok=True)
    if os.path.islink(dst): os.unlink(dst)
    elif os.path.exists(dst):
        import shutil; shutil.rmtree(dst)
    os.symlink(src, dst)

# Inference deps: model libs + retrieval libs (bm25s, sentence-transformers) +
# config libs (omegaconf, pyyaml). Mirrors notebook 41 cell 5's install set.
!pip install -q --upgrade \
    "peft>=0.10" "transformers>=4.40" "accelerate>=0.30" "torchao>=0.17" \
    "bm25s>=0.3.0,<0.4" "sentence-transformers" \
    "datasets" "pandas<3.0" "tqdm" "omegaconf" "pyyaml" \
    "trl>=0.12.0"


In [ ]:
# 2) Run config 180 on Blind-A.
%cd /content/recsys2026/music-crs-baselines
!python run_inference_blindset.py \
    --tid 180-wrrf-bge-m3-ft-v5kto-blindA \
    --batch_size 32 \
    2>&1 | tee /content/drive/MyDrive/recsys2026_retrieval_v2_cache/blindA_180_log.txt | tail -60

In [ ]:
# 3) Validate the prediction.json: must be 80 entries (80 unique sessions × 1 turn each).
%cd /content/recsys2026
import json
pred_path = 'music-crs-baselines/exp/inference/blindset_A/180-wrrf-bge-m3-ft-v5kto-blindA.json'
preds = json.load(open(pred_path))
n_entries = len(preds) if isinstance(preds, list) else len(preds.keys())
print(f'prediction file: {n_entries} entries')
assert n_entries == 80, f'EXPECTED 80, got {n_entries} — DO NOT submit'
sample_keys = list(preds[0].keys()) if isinstance(preds, list) else list(list(preds.values())[0].keys())
print(f'sample entry keys: {sample_keys}')

In [ ]:
# 4) Validate per existing validator (catches schema bugs before submission).
# Use subprocess + assert rc==0 so a failing precheck or validator HALTS the cell
# (IPython's `!` doesn't propagate non-zero exit codes — bad payloads would slip
# through to the zip step otherwise).
%cd /content/recsys2026
import subprocess
import sys
sys.path.insert(0, '/content/recsys2026')
from scripts.precheck_prediction import precheck, _load_catalog
from pathlib import Path

PRED_PATH = Path('music-crs-baselines/exp/inference/blindset_A/180-wrrf-bge-m3-ft-v5kto-blindA.json')

# Stricter precheck — call directly (avoids subprocess + redundant catalog load).
catalog = _load_catalog("talkpl-ai/TalkPlayData-Challenge-Track-Metadata")
result = precheck(PRED_PATH, catalog=catalog, expected_n=80)
assert result['ok'], (
    f'precheck FAILED with {len(result["errors"])} errors; '
    f'first 5: {result["errors"][:5]}'
)
print(f'OK: precheck passed (n_records={result["n_records"]})')

# Schema validator (less strict but covers different bugs).
# --split blindA is REQUIRED (was missing in v1 of this cell; rc=2 from argparse on missing arg).
rc = subprocess.call(['python', 'scripts/validate_prediction.py', '--input', str(PRED_PATH), '--split', 'blindA'])
assert rc == 0, f'validate_prediction.py FAILED (rc={rc}) — do not submit'
print('OK: schema validator passed')

In [ ]:
# 5) Zip for CodaBench (prediction.json must be at the ROOT of the zip).
import os, zipfile, datetime
date_str = datetime.date.today().strftime('%Y-%m-%d')
zip_path = f'/content/drive/MyDrive/recsys2026_submissions/{date_str}-retrieval-v2-180-bge-m3-ft.zip'
os.makedirs(os.path.dirname(zip_path), exist_ok=True)
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    z.write(pred_path, arcname='prediction.json')
print(f'zip ready: {zip_path}')
print(f'size: {os.path.getsize(zip_path) / 1024:.1f} KB')

## After the run

1. Download the zip from Drive: `/content/drive/MyDrive/recsys2026_submissions/<date>-retrieval-v2-180-bge-m3-ft.zip`
2. Upload to CodaBench (https://www.codabench.org/competitions/).
3. Append the composite + nDCG@20 + LLM + lex_div + cat_div scores to the score tracker:
   ```
   !python scripts/blind_a_score_tracker.py append --tid 180-wrrf-bge-m3-ft-v5kto-blindA \
       --composite <X> --ndcg <Y> --llm <Z> --lex_div <W> --cat_div <V>
   ```
4. **If composite ≥ 0.21** (Submission 1 gate, spec §9): retrieval v2 Stage A is
   shippable. Proceed to Stage B (cross-encoder fine-tune, nb 71).
5. **If composite < 0.21**: check the axis breakdown — if nDCG dropped, the
   fine-tune regressed and we need to debug; if LLM/lex_div tanked, the
   responder context got polluted by the new top-N candidates (rare on a
   retrieval-only change). Consult
   `docs/superpowers/specs/2026-05-18-ndcg-stretch-design.md` §10 for abort rules.